In [0]:
# =============================================================================
# QUALIDADE DE DADOS - Análise da camada Bronze
# =============================================================================
# Objetivo: Verificar a qualidade dos dados brutos antes das transformações,
# identificando problemas de completude, consistência, unicidade e acurácia.
# Essa análise justifica as decisões de limpeza tomadas na camada Silver.
# =============================================================================

from pyspark.sql import functions as F

# Função auxiliar que analisa qualidade de qualquer tabela
def analisar_qualidade(nome_tabela):
    df = spark.table(f"workspace.default.{nome_tabela}")
    total = df.count()
    
    print(f"\n{'='*60}")
    print(f"TABELA: {nome_tabela} | Total de registros: {total:,}")
    print(f"{'='*60}")
    
    # Completude: verifica nulos por coluna
    print("\n--- COMPLETUDE (valores nulos por coluna) ---")
    houve_nulo = False
    for col in df.columns:
        if col.startswith("_"):
            continue  # ignora colunas de metadado
        nulos = df.filter(F.col(col).isNull()).count()
        pct = round((nulos / total) * 100, 2)
        if nulos > 0:
            print(f"  {col}: {nulos:,} nulos ({pct}%)")
            houve_nulo = True
    if not houve_nulo:
        print("  Nenhum valor nulo encontrado.")
    
    # Unicidade: verifica duplicatas na primeira coluna identificadora
    id_col = [c for c in df.columns if c.endswith("_id") and not c.startswith("_")]
    if id_col:
        id_col = id_col[0]
        duplicatas = total - df.dropDuplicates([id_col]).count()
        print(f"\n--- UNICIDADE ---")
        print(f"  Duplicatas por {id_col}: {duplicatas:,}")

# Analisa todas as tabelas Bronze que alimentam as camadas seguintes
for tabela in [
    "bronze_orders",
    "bronze_order_items", 
    "bronze_order_payments",
    "bronze_order_reviews",
    "bronze_products",
    "bronze_customers",
    "bronze_sellers",
    "bronze_category_translation"
]:
    analisar_qualidade(tabela)

In [0]:
# =============================================================================
# QUALIDADE DE DADOS - Verificações de acurácia e consistência
# =============================================================================

print("="*60)
print("ACURÁCIA - Valores fora do domínio esperado")
print("="*60)

items = spark.table("workspace.default.bronze_order_items")
payments = spark.table("workspace.default.bronze_order_payments")
orders = spark.table("workspace.default.bronze_orders")
reviews = spark.table("workspace.default.bronze_order_reviews")

# Preços negativos ou zerados em order_items
preco_invalido = items.filter(F.col("price") <= 0).count()
print(f"\nbronze_order_items:")
print(f"  Registros com preço <= 0: {preco_invalido:,}")

# Frete negativo
frete_invalido = items.filter(F.col("freight_value") < 0).count()
print(f"  Registros com frete negativo: {frete_invalido:,}")

# Pagamentos com valor negativo
pagamento_invalido = payments.filter(F.col("payment_value") < 0).count()
print(f"\nbronze_order_payments:")
print(f"  Pagamentos com valor negativo: {pagamento_invalido:,}")

# Formas de pagamento disponíveis (consistência de categorias)
print(f"\n  Formas de pagamento encontradas:")
payments.groupBy("payment_type").count().orderBy("count", ascending=False).show()

# Status de pedidos disponíveis
print(f"bronze_orders:")
print(f"  Status de pedidos encontrados:")
orders.groupBy("order_status").count().orderBy("count", ascending=False).show()

# =============================================================================
# ACURÁCIA - Domínio do campo review_score
# =============================================================================
# O campo review_score deve conter apenas inteiros de 1 a 5 (escala de
# avaliação da plataforma). Valores fora desse domínio indicam corrupção
# de dados na origem ou erro de leitura do arquivo.
# =============================================================================

print("="*60)
print("ACURÁCIA - Domínio do campo review_score")
print("="*60)

total_reviews = reviews.count()

# try_cast converte para inteiro tolerando valores malformados (retorna null)
reviews_check = reviews.withColumn(
    "score_int",
    F.expr("try_cast(review_score as int)")
)

# Valores que não convertem para inteiro
nao_conversiveis = reviews_check.filter(F.col("score_int").isNull()).count()

# Valores que convertem mas estão fora da escala válida de 1 a 5
fora_dominio = reviews_check.filter(
    F.col("score_int").isNotNull() & ~F.col("score_int").between(1, 5)
).count()

validos = total_reviews - nao_conversiveis - fora_dominio

print(f"\nbronze_order_reviews:")
print(f"  Total de registros:                    {total_reviews:,}")
print(f"  Valores não conversíveis para inteiro: {nao_conversiveis:,}")
print(f"  Valores fora da escala de 1 a 5:       {fora_dominio:,}")
print(f"  Registros válidos:                     {validos:,}")

print(f"\n  Distribuição das notas válidas:")
(reviews_check
    .filter(F.col("score_int").between(1, 5))
    .groupBy("score_int")
    .count()
    .orderBy("score_int")
    .show())

# --- OUTLIERS: valores extremos em preço ---
print("="*60)
print("OUTLIERS - Estatísticas de preço e frete")
print("="*60)
items.select(
    F.round(F.min("price"), 2).alias("preco_min"),
    F.round(F.avg("price"), 2).alias("preco_medio"),
    F.round(F.max("price"), 2).alias("preco_max"),
    F.round(F.min("freight_value"), 2).alias("frete_min"),
    F.round(F.avg("freight_value"), 2).alias("frete_medio"),
    F.round(F.max("freight_value"), 2).alias("frete_max"),
).show()

In [0]:
# =============================================================================
# QUALIDADE DE DADOS - Resumo consolidado dos problemas encontrados
# =============================================================================

print("""
RESUMO DA ANÁLISE DE QUALIDADE DE DADOS
========================================

0. PROBLEMA CRÍTICO NA INGESTÃO
--------------------------------------------
O achado mais relevante desta análise não estava no conteúdo dos dados,
mas na forma como eram lidos.

Sintoma: erro de conversão ao tentar tipar o campo review_score, com
valores de texto aparecendo em colunas de data e datas aparecendo na
coluna de nota.

Causa: o arquivo olist_order_reviews_dataset.csv contém comentários
escritos por clientes, campos de texto livre com quebras de linha e
vírgulas. Com as opções padrão de leitura de CSV do Spark, cada quebra
de linha dentro de um comentário era interpretada como fim de registro,
gerando linhas fantasma com colunas desalinhadas.

Correção: reingestão com as opções multiLine, quote e escape
configuradas no notebook 01_bronze_ingestion.

Impacto: o total de registros caiu de 104.162 para 99.224. Os 4.938
registros eliminados eram integralmente compostos por linhas inválidas.

Validação da correção: após a reingestão, a verificação de domínio do
campo review_score retornou zero valores não conversíveis e zero valores
fora da escala de 1 a 5. Os 99.224 registros são integralmente válidos,
confirmando que o problema foi eliminado na origem e não apenas mascarado
nas camadas seguintes.

Observação metodológica: este problema passaria despercebido em uma
inspeção superficial de completude ou unicidade, pois as linhas
fantasma não eram nulas nem duplicadas. Teria contaminado silenciosamente
a análise da Pergunta 6.

1. COMPLETUDE
-------------
bronze_orders:
  - order_approved_at: 160 nulos (0.16%)
    Tratamento: mantidos. Pedidos sem aprovação podem ser cancelados ou
    em processamento. Não impacta as análises de padrão de compra.
  - order_delivered_carrier_date: 1.783 nulos (1.79%)
  - order_delivered_customer_date: 2.965 nulos (2.98%)
    Tratamento: mantidos. Pedidos ainda em trânsito ou cancelados
    naturalmente não têm data de entrega.

bronze_products:
  - product_category_name: 610 nulos (1.85%)
    Tratamento: removidos na Silver. Produtos sem categoria não contribuem
    para análises por categoria, que é o foco do objetivo.
  - Demais campos descritivos (nome, descrição, fotos, dimensões):
    610 a 2 nulos. Tratamento: mantidos, pois não são usados na Gold.

bronze_order_reviews:
  - review_comment_title: 87.656 nulos (88.34%)
  - review_comment_message: 58.247 nulos (58.70%)
    Tratamento: mantidos. O preenchimento de comentário é opcional na
    plataforma, portanto a ausência é comportamento esperado e não
    defeito. Campos não utilizados nas análises deste MVP.

bronze_customers, bronze_sellers, bronze_category_translation,
bronze_order_items, bronze_order_payments:
  - Sem valores nulos. Dados completos.

2. UNICIDADE
------------
bronze_order_reviews: 814 duplicatas por review_id
  PROBLEMA REAL. Diferente dos casos abaixo, review_id deveria ser
  único por definição, já que representa uma avaliação individual.
  A presença de duplicatas indica falha de integridade na origem,
  provavelmente reenvio de formulário ou erro de exportação.
  Tratamento: deduplicação por review_id na camada Silver, resultando
  em 98.410 registros únicos.

bronze_order_items: 13.984 duplicatas por order_id
  Falso positivo. Um pedido pode ter múltiplos itens, portanto a chave
  única é a combinação order_id + order_item_id, não o order_id isolado.
  Nenhum tratamento necessário além da deduplicação pela chave composta.

bronze_order_payments: 4.446 duplicatas por order_id
  Falso positivo. Um pedido pode ter múltiplas formas de pagamento
  (ex: cartão mais voucher). Tratamento: agregados por order_id na
  construção da tabela Gold para preservar a granularidade da fato.

bronze_orders, bronze_products, bronze_customers, bronze_sellers:
  Sem duplicatas nas chaves primárias.

3. ACURÁCIA
-----------
bronze_order_items:
  - Sem preços negativos ou zerados.
  - Frete mínimo de R$ 0,00: possível para retirada em loja ou promoção.
    Mantido como válido.
  - Preço máximo de R$ 6.735,00: valor alto mas plausível para eletrônicos
    ou itens premium. Não caracteriza erro.

bronze_order_payments:
  - 3 registros com payment_type = "not_defined". Volume irrelevante
    (0.003% do total). Mantidos na Silver, excluídos apenas na análise
    da Pergunta 3.
  - Sem valores de pagamento negativos.

bronze_order_reviews:
  - Campo review_score integralmente dentro do domínio válido após a
    correção de ingestão. Distribuição: nota 1 com 11.424 registros,
    nota 2 com 3.151, nota 3 com 8.179, nota 4 com 19.142 e nota 5 com
    57.328. A concentração em notas altas é consistente com o padrão
    típico de avaliações em plataformas de e-commerce.
  - O uso de try_cast na camada Silver foi mantido como salvaguarda
    defensiva, garantindo que eventuais valores malformados em cargas
    futuras não interrompam o pipeline.

4. CONSISTÊNCIA
---------------
  - Estados de clientes e vendedores padronizados para maiúsculo na Silver
    (customer_state, seller_state) para evitar inconsistências como
    "sp" vs "SP".
  - Datas convertidas de string para Timestamp na Silver para permitir
    operações temporais corretas nas análises.
  - Categorias de produto padronizadas via join com a tabela oficial de
    tradução, garantindo nomenclatura consistente.

CONCLUSÃO
---------
Foram identificados dois problemas reais de qualidade neste dataset.

O primeiro, e mais grave, não estava no conteúdo dos dados mas na sua
ingestão: a leitura incorreta de campos de texto multilinha no arquivo
de reviews gerava registros corrompidos que teriam contaminado
silenciosamente a análise da Pergunta 6. A correção foi aplicada na
camada Bronze e validada posteriormente com verificação de domínio.

O segundo são as 814 duplicatas de review_id, resolvidas por
deduplicação na camada Silver.

Os demais achados são falsos positivos que refletem a granularidade
legítima das tabelas, ou ausências esperadas em campos de preenchimento
opcional. Nenhum valor negativo, categoria inconsistente ou outlier
espúrio foi encontrado nos campos financeiros.

O caso do arquivo de reviews reforça o papel da camada Bronze como ponto
de rastreabilidade: a correção foi feita na origem e propagada por
reprocessamento, sem necessidade de ajustes pontuais nas camadas
seguintes.
""")